# TitanAgent 実験3.5: フォーマット先行習得 ＋ 長さカリキュラム SFT on Titans (MAC)

実験 3 の結果（`stream_token_cot` の未見長距離 `d16` でのフォーマット崩れ、`ephemeral_token_cot` の後半ターン学習不足）を踏まえた改良版です。

### 実験 3.5 の 3 つの改良点
1. **2 段階 SFT（フォーマット先行習得 → 長さ混合カリキュラム）**:
   - **Stage 1（前半 25%）**: 短く単純な問題（1〜2 ターン, `d=1..4`）で `<think>` / `<t>...</t>` / `<ans>...</ans>` の型と 5 段ルールを完璧に叩き込む。
   - **Stage 2（後半 75%）**: 短いアンカー（25%）＋長距離単発（`d=8..22`, 35%）＋多ターン長ストリーム（最大 6 ターン, 40%）を混合し、**短〜超長まで幅広く汎化**させる。
2. **`ephemeral_token_cot` の後半ターン重み付けサンプリング**:
   - 実験 3 で起きた「Turn 1 が 55.8%・Turn 3 が 13.3%」という偏りを解消し、後半ターンほど重み付けして均等に学習。
3. **超長期ベンチマーク＆フォーマット診断指標（`format_ok`）の追加**:
   - 5 ベンチマーク（`single_turn_d4`, `long_range_d16`, `ultra_long_d28` [未見超長距離], `multi_turn_3t_d6`, **`multi_turn_6t_d8`** [6 ターン連続]）で評価。
   - 「フォーマット崩れ」と「記憶・推論ミス」を分離して観測可能。

> **重要**: Kaggle で実行する際は、右側パネルの **Accelerator から `GPU T4 x2`（または `P100`）を必ず選択**してください（GPU T4 で約 30〜45 分目安）。

In [ ]:
# 1. 環境確認 & ライブラリインストール
import os, sys, torch

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
print(f'Environment: Colab={IN_COLAB}, Kaggle={IN_KAGGLE}')
print(f'PyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ WARNING: CUDA が無効（CPU 実行）です！Kaggle 右側パネルの Accelerator から GPU (T4 / P100) を選択してください。')

!pip -q install -U titans-pytorch

In [ ]:
# 2. 実験設定（Kaggle T4 16GB OOM 安全デフォルト）
from pathlib import Path

PRESET = 'tiny'              # 'micro' (~1.4M), 'tiny' (推奨・~10.1M), 'small' (~22M)
MODEL_TYPE = 'titans'        # 'titans' または比較用 'sliding_transformer'
MODES = ['turn_cot', 'stream_token_cot', 'ephemeral_token_cot']
SFT_STEPS = 400              # turn_cot / stream_token_cot の総学習ステップ数
STAGE1_RATIO = 0.25          # 前半 25% をフォーマット先行習得（Stage 1）に使用
EPHEMERAL_STEP_MULT = 2.5    # ephemeral は1シーケンス1ホップ学習のためステップ倍率を設定
EVAL_EPISODES = 25           # 各ベンチマークの評価問題数
LOG_EVERY = 50
SEED = 42

if IN_KAGGLE:
    OUTPUT_DIR = Path('/kaggle/working/checkpoints/exp3_5_length_curriculum')
else:
    OUTPUT_DIR = Path('./checkpoints/exp3_5_length_curriculum')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PRESETS = {
    'micro': dict(dim=128, depth=3, segment_len=16, num_persist_mem_tokens=4, num_longterm_mem_tokens=8,  seq_len=1024, batch_size=4, grad_accum=2,  lr=1.5e-3),
    'tiny':  dict(dim=256, depth=4, segment_len=32, num_persist_mem_tokens=4, num_longterm_mem_tokens=16, seq_len=1024, batch_size=4, grad_accum=8,  lr=8e-4),
    'small': dict(dim=384, depth=6, segment_len=32, num_persist_mem_tokens=8, num_longterm_mem_tokens=16, seq_len=1024, batch_size=2, grad_accum=16, lr=5e-4),
}
cfg = PRESETS[PRESET]
print('OUTPUT_DIR =', OUTPUT_DIR)
print('Selected preset:', PRESET, cfg)

In [ ]:
# 3. 意味単位トークナイザー & 5段ルール推論・マルチターン生成器
import json, math, random, time
from contextlib import nullcontext
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F

SPECIAL_TOKENS = [
    '<|pad|>', '<|eos|>', '<|unk|>',
    '<think>', '</think>', '<t>', '</t>',
    '<ans>', '</ans>', '<turn>', '</turn>',
]
SLOT_NAMES = ['時間', '予算', '優先', '経験', '目標']
SLOT_VALUES = {
    '時間': ['極短', '短', '中', '週末'],
    '予算': ['無料', '小額', '十分'],
    '優先': ['継続', '速効', '品質', '低負荷'],
    '経験': ['初級', '中級', '上級'],
    '目標': ['副収入', '転職', '資格', '制作', '習慣'],
}
TOPICS = ['アプリ', '英語', 'ブログ', 'デザイン', '自動化', '資格学習', '発信', '研究']
ANSWER_SLOTS = ['負荷', 'ツール', 'ペース', '検証', '戦略']
ANSWER_VALUES = {
    '負荷': ['負荷:軽', '負荷:中', '負荷:重'],
    'ツール': ['ツール:無料', 'ツール:AI課金', 'ツール:教材購入'],
    'ペース': ['ペース:毎日小刻み', 'ペース:平日固定', 'ペース:週末集中', 'ペース:短期スプリント'],
    '検証': ['検証:自己チェック', '検証:公開テスト', '検証:定量計測', '検証:レビュー依頼'],
    '戦略': ['戦略:最小継続型', '戦略:一点突破型', '戦略:設計重視型', '戦略:低コスト検証型', '戦略:週末集中型', '戦略:高速反復型'],
}
DISTRACTOR_EVENTS = [
    ['メモ:', '進捗', '確認中'], ['メモ:', '少し', '休憩'],
    ['メモ:', '別件', '整理中'], ['メモ:', '課題', '見直し'],
    ['メモ:', '環境', '調整済'], ['メモ:', '優先度', '検討中'],
    ['メモ:', '資料', '調査中'], ['メモ:', '記録', '保存済'],
]
STRUCTURAL_TOKENS = [
    '設定:', '更新:', 'メモ:', '相談:', '記憶', '現在', '=>', '|',
    '進捗', '確認中', '少し', '休憩', '別件', '整理中', '課題', '見直し',
    '環境', '調整済', '優先度', '検討中', '資料', '調査中', '記録', '保存済',
]

class SemanticTokenizer:
    def __init__(self, token_to_id):
        self.token_to_id = dict(token_to_id)
        self.id_to_token = {v: k for k, v in self.token_to_id.items()}
        self.pad_token_id = self.token_to_id['<|pad|>']
        self.eos_token_id = self.token_to_id['<|eos|>']
        self.unk_token_id = self.token_to_id['<|unk|>']
        self.ans_start_id = self.token_to_id['<ans>']
        self.ans_end_id = self.token_to_id['</ans>']
        self.end_turn_id = self.token_to_id['</turn>']
        self.vocab_size = len(self.token_to_id)

    @classmethod
    def build_default(cls):
        vocab = {}
        def _add(tok):
            if tok not in vocab:
                vocab[tok] = len(vocab)
        for tok in SPECIAL_TOKENS + STRUCTURAL_TOKENS + TOPICS:
            _add(tok)
        for slot in SLOT_NAMES:
            _add(slot)
            for val in SLOT_VALUES[slot]:
                _add(f'{slot}={val}')
        for ans_slot in ANSWER_SLOTS:
            for val in ANSWER_VALUES[ans_slot]:
                _add(val)
        return cls(vocab)

    def encode_tokens(self, tokens):
        return [self.token_to_id.get(t, self.unk_token_id) for t in tokens]

    def decode_tokens(self, ids, skip_pad=True):
        return [self.id_to_token.get(i, '<|unk|>') for i in ids if not (skip_pad and i == self.pad_token_id)]

def solve_rules(slots):
    t_val, b_val, p_val, e_val, g_val = slots['時間'], slots['予算'], slots['優先'], slots['経験'], slots['目標']
    if t_val == '極短' or p_val == '低負荷':
        y1 = '負荷:軽'
    elif (p_val == '速効' and t_val in ('中', '週末')) or (p_val == '品質' and t_val == '週末'):
        y1 = '負荷:重'
    else:
        y1 = '負荷:中'
    r1 = ['記憶', f'時間={t_val}', f'優先={p_val}', '=>', y1]

    if b_val == '無料':
        y2 = 'ツール:無料'
    elif (b_val == '十分' and e_val in ('中級', '上級')) or (b_val == '小額' and e_val == '上級'):
        y2 = 'ツール:AI課金'
    else:
        y2 = 'ツール:教材購入'
    r2 = ['記憶', f'予算={b_val}', f'経験={e_val}', '=>', y2]

    if t_val == '週末':
        y3 = 'ペース:週末集中'
    elif y1 == '負荷:重' and p_val == '速効':
        y3 = 'ペース:短期スプリント'
    elif y1 == '負荷:軽' or p_val == '継続':
        y3 = 'ペース:毎日小刻み'
    else:
        y3 = 'ペース:平日固定'
    r3 = ['現在', y1, f'時間={t_val}', f'優先={p_val}', '=>', y3]

    if g_val in ('副収入', '制作') and e_val != '初級':
        y4 = '検証:公開テスト'
    elif g_val in ('転職', '資格') and y2 == 'ツール:教材購入':
        y4 = '検証:定量計測'
    elif y2 == 'ツール:AI課金':
        y4 = '検証:レビュー依頼'
    else:
        y4 = '検証:自己チェック'
    r4 = ['現在', y2, f'目標={g_val}', f'経験={e_val}', '=>', y4]

    if y3 == 'ペース:短期スプリント':
        y5 = '戦略:一点突破型'
    elif y3 == 'ペース:週末集中':
        y5 = '戦略:週末集中型'
    elif y4 == '検証:公開テスト' and y1 != '負荷:軽':
        y5 = '戦略:高速反復型'
    elif p_val == '品質' and y4 in ('検証:定量計測', '検証:レビュー依頼'):
        y5 = '戦略:設計重視型'
    elif y1 == '負荷:軽':
        y5 = '戦略:最小継続型'
    else:
        y5 = '戦略:低コスト検証型'
    r5 = ['現在', y1, y3, y4, f'優先={p_val}', '=>', y5]
    return [y1, y2, y3, y4, y5], [r1, r2, r3, r4, r5]

@dataclass
class TurnSpec:
    turn_idx: int
    topic: str
    prefix_tokens: list
    slots: dict
    stale_slots: dict
    answers: list
    thoughts: list
    stale_answers: object

@dataclass
class MultiTurnEpisode:
    turns: list
    num_distractors_per_turn: int

def _make_distractor_tokens(rng, count):
    toks = []
    for _ in range(count):
        toks.extend(rng.choice(DISTRACTOR_EVENTS))
    return toks

def generate_multiturn_episode(rng, num_turns=3, num_distractors=6, updates_per_turn=1):
    slots = {k: rng.choice(SLOT_VALUES[k]) for k in SLOT_NAMES}
    topics = rng.sample(TOPICS, k=min(num_turns, len(TOPICS)))
    while len(topics) < num_turns:
        topics.append(rng.choice(TOPICS))
    turns = []
    for t_idx in range(num_turns):
        prefix = []
        if t_idx == 0:
            prefix.append('設定:')
            for k in SLOT_NAMES:
                prefix.append(f'{k}={slots[k]}')
        d_before = num_distractors // 2
        d_after = num_distractors - d_before
        prefix.extend(_make_distractor_tokens(rng, d_before))
        stale_slots = {}
        if updates_per_turn > 0:
            for k in rng.sample(SLOT_NAMES, k=min(updates_per_turn, len(SLOT_NAMES))):
                old_val = slots[k]
                new_val = rng.choice([v for v in SLOT_VALUES[k] if v != old_val])
                stale_slots[k] = old_val
                slots[k] = new_val
                prefix.extend(['更新:', f'{k}={new_val}'])
        prefix.extend(_make_distractor_tokens(rng, d_after))
        prefix.extend(['<turn>', '相談:', topics[t_idx]])
        answers, thoughts = solve_rules(slots)
        stale_answers = None
        if stale_slots:
            sc = dict(slots); sc.update(stale_slots)
            stale_answers, _ = solve_rules(sc)
        turns.append(TurnSpec(t_idx, topics[t_idx], prefix, dict(slots), stale_slots, answers, thoughts, stale_answers))
    return MultiTurnEpisode(turns=turns, num_distractors_per_turn=num_distractors)

def format_turn_completion_tokens(turn, mode):
    if mode == 'turn_cot':
        toks = ['<think>']
        for i, r in enumerate(turn.thoughts):
            if i > 0: toks.append('|')
            toks.extend(r)
        return toks + ['</think>', '<ans>', *turn.answers, '</ans>', '</turn>']
    if mode == 'stream_token_cot':
        toks = ['<ans>']
        for r, y in zip(turn.thoughts, turn.answers):
            toks.extend(['<t>', *r, '</t>', y])
        return toks + ['</ans>', '</turn>']
    if mode == 'ephemeral_token_cot':
        return ['<ans>', *turn.answers, '</ans>', '</turn>']
    raise ValueError(mode)

tokenizer = SemanticTokenizer.build_default()
print('Semantic vocab size:', tokenizer.vocab_size)

In [ ]:
# 4. 2段階カリキュラム SFT (フォーマット先行習得 -> 長さ混合) & モデル構築
def _sample_balanced_turn_idx(num_turns, rng):
    if num_turns <= 1:
        return 0
    weights = [i + 1 for i in range(num_turns)]
    return rng.choices(range(num_turns), weights=weights, k=1)[0]

def build_sft_example(ep, mode, tokenizer, max_seq_len, rng):
    if mode in ('turn_cot', 'stream_token_cot'):
        seq_toks, mask = [], []
        for t in ep.turns:
            seq_toks.extend(t.prefix_tokens); mask.extend([0] * len(t.prefix_tokens))
            comp = format_turn_completion_tokens(t, mode)
            seq_toks.extend(comp); mask.extend([1] * len(comp))
        seq_toks.append('<|eos|>'); mask.append(1)
        ids = tokenizer.encode_tokens(seq_toks)
        labels = [tid if m == 1 else -100 for tid, m in zip(ids, mask)]
        return ids[:max_seq_len], labels[:max_seq_len]

    if mode == 'ephemeral_token_cot':
        target_t_idx = _sample_balanced_turn_idx(len(ep.turns), rng)
        target_hop = rng.randrange(len(ANSWER_SLOTS))
        prefix_toks = []
        for idx in range(target_t_idx):
            tp = ep.turns[idx]
            prefix_toks.extend(tp.prefix_tokens)
            prefix_toks.extend(format_turn_completion_tokens(tp, 'ephemeral_token_cot'))
        tc = ep.turns[target_t_idx]
        prefix_toks.extend(tc.prefix_tokens)
        prefix_toks.append('<ans>')
        prefix_toks.extend(tc.answers[:target_hop])
        target_toks = ['<t>', *tc.thoughts[target_hop], '</t>', tc.answers[target_hop], '<|eos|>']
        p_ids = tokenizer.encode_tokens(prefix_toks)
        t_ids = tokenizer.encode_tokens(target_toks)
        ids = p_ids + t_ids
        labels = [-100] * len(p_ids) + list(t_ids)
        if len(ids) > max_seq_len:
            ov = len(ids) - max_seq_len
            ids, labels = ids[ov:], labels[ov:]
        return ids, labels
    raise ValueError(mode)

def sample_curriculum_episode(rng, progress, stage1_ratio=0.25):
    if progress < stage1_ratio:
        num_turns = rng.choices([1, 2], weights=[0.60, 0.40])[0]
        num_dist = rng.randint(1, 4)
        updates = rng.choices([0, 1], weights=[0.30, 0.70])[0]
        return generate_multiturn_episode(rng, num_turns=num_turns, num_distractors=num_dist, updates_per_turn=updates), 'S1_fmt'

    s2_prog = (progress - stage1_ratio) / max(1e-6, 1.0 - stage1_ratio)
    bucket = rng.random()
    if bucket < 0.25:
        num_turns = rng.choices([1, 2], weights=[0.5, 0.5])[0]
        num_dist = rng.randint(2, 6)
    elif bucket < 0.60:
        num_turns = 1
        max_d = 14 if s2_prog < 0.4 else 22
        num_dist = rng.randint(8, max_d)
    else:
        if s2_prog < 0.35:
            num_turns = rng.choices([2, 3, 4], weights=[0.25, 0.45, 0.30])[0]
            num_dist = rng.randint(4, 8)
        else:
            num_turns = rng.choices([3, 4, 5, 6], weights=[0.20, 0.25, 0.25, 0.30])[0]
            num_dist = rng.randint(4, 10)
    updates = rng.choices([0, 1], weights=[0.15, 0.85])[0]
    return generate_multiturn_episode(rng, num_turns=num_turns, num_distractors=num_dist, updates_per_turn=updates), 'S2_len'

class FallbackCausalTransformer(nn.Module):
    def __init__(self, num_tokens, dim, depth, max_seq_len=1280, window_size=0):
        super().__init__()
        self.window_size = window_size
        self.tok_emb = nn.Embedding(num_tokens, dim)
        self.pos_emb = nn.Embedding(max_seq_len, dim)
        layer = nn.TransformerEncoderLayer(d_model=dim, nhead=max(1, dim // 64), dim_feedforward=dim * 4, dropout=0.0, activation='gelu', batch_first=True, norm_first=True)
        self.blocks = nn.TransformerEncoder(layer, num_layers=depth)
        self.ln_f = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, num_tokens, bias=False)

    def forward(self, x):
        bsz, seqlen = x.shape
        pos = torch.arange(seqlen, device=x.device).unsqueeze(0).expand(bsz, seqlen)
        h = self.tok_emb(x) + self.pos_emb(pos)
        mask = torch.triu(torch.ones(seqlen, seqlen, device=x.device, dtype=torch.bool), diagonal=1)
        if self.window_size > 0:
            mask = mask | torch.tril(torch.ones(seqlen, seqlen, device=x.device, dtype=torch.bool), diagonal=-self.window_size)
        return self.head(self.ln_f(self.blocks(h, mask=mask)))

def build_model(model_type, model_cfg, device):
    if model_type == 'titans':
        from titans_pytorch import MemoryAsContextTransformer
        return MemoryAsContextTransformer(
            num_tokens=model_cfg['num_tokens'], dim=model_cfg['dim'], depth=model_cfg['depth'],
            segment_len=model_cfg['segment_len'], num_persist_mem_tokens=model_cfg['num_persist_mem_tokens'],
            num_longterm_mem_tokens=model_cfg['num_longterm_mem_tokens'],
        ).to(device)
    win = model_cfg['segment_len'] if model_type == 'sliding_transformer' else 0
    return FallbackCausalTransformer(model_cfg['num_tokens'], model_cfg['dim'], model_cfg['depth'], max(1280, model_cfg['seq_len'] + 128), win).to(device)

def collate_batch(items, pad_id):
    max_len = max(len(ids) for ids, _ in items)
    x = torch.full((len(items), max_len), pad_id, dtype=torch.long)
    y = torch.full((len(items), max_len), -100, dtype=torch.long)
    for i, (ids, lbls) in enumerate(items):
        x[i, :len(ids)] = torch.tensor(ids, dtype=torch.long)
        y[i, :len(lbls)] = torch.tensor(lbls, dtype=torch.long)
    return x, y

def cosine_lr(step, warmup, total, max_lr, min_ratio=0.1):
    if step < warmup: return max_lr * (step + 1) / max(1, warmup)
    p = min(max((step - warmup) / max(1, total - warmup), 0.0), 1.0)
    return max_lr * (min_ratio + (1.0 - min_ratio) * 0.5 * (1.0 + math.cos(math.pi * p)))

def train_mode(mode, model_type, model_cfg, tokenizer, device, amp_dtype, steps, stage1_ratio, batch_size, grad_accum, lr, log_every=50, seed=42):
    torch.manual_seed(seed)
    rng = random.Random(seed)
    model = build_model(model_type, model_cfg, device)
    n_params = sum(p.numel() for p in model.parameters())
    use_amp = device.type == 'cuda' and amp_dtype in (torch.bfloat16, torch.float16)
    use_scaler = device.type == 'cuda' and amp_dtype == torch.float16
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.05)
    scaler = torch.amp.GradScaler('cuda', enabled=use_scaler) if hasattr(torch.amp, 'GradScaler') else torch.cuda.amp.GradScaler(enabled=use_scaler)
    warmup = max(1, min(40, steps // 10))

    model.train()
    t0 = time.time()
    print(f'\n=== [Train: {mode}] params={n_params/1e6:.2f}M | steps={steps} (S1={int(steps*stage1_ratio)}, S2={steps-int(steps*stage1_ratio)}) | batch={batch_size}x{grad_accum} ===')
    final_loss = 0.0
    for step in range(1, steps + 1):
        progress = (step - 1) / max(1, steps)
        cur_lr = cosine_lr(step - 1, warmup, steps, lr)
        for pg in optimizer.param_groups: pg['lr'] = cur_lr
        optimizer.zero_grad(set_to_none=True)
        accum_loss, step_lens = 0.0, []
        stage_tag = 'S1_fmt'
        for _ in range(grad_accum):
            items = []
            for _ in range(batch_size):
                ep, stage_tag = sample_curriculum_episode(rng, progress, stage1_ratio)
                items.append(build_sft_example(ep, mode, tokenizer, model_cfg['seq_len'], rng))
            step_lens.extend(len(ids) for ids, _ in items)
            x, y = collate_batch(items, tokenizer.pad_token_id)
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            ctx = torch.amp.autocast(device_type=device.type, dtype=amp_dtype) if use_amp else nullcontext()
            with ctx:
                logits = model(x[:, :-1])
                loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y[:, 1:].reshape(-1), ignore_index=-100)
                scaled = loss / grad_accum
            if use_scaler: scaler.scale(scaled).backward()
            else: scaled.backward()
            accum_loss += float(loss.item()) / grad_accum
        if use_scaler:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        final_loss = accum_loss
        if step % log_every == 0 or step == 1 or step == steps:
            print(f'  [{mode} | {stage_tag}] step {step:4d}/{steps} | loss {accum_loss:.4f} | avg_len {sum(step_lens)/len(step_lens):.1f} | lr {cur_lr:.2e} | {time.time()-t0:.1f}s')
    torch.save({'mode': mode, 'steps': steps, 'model_state_dict': model.state_dict()}, OUTPUT_DIR / f'{mode}_latest.pt')
    if device.type == 'cuda': torch.cuda.empty_cache()
    return model, {'steps': steps, 'final_loss': round(final_loss, 4), 'train_seconds': round(time.time() - t0, 1)}

@torch.no_grad()
def _greedy_generate_until(model, prefix_ids, stop_ids, max_new_tokens, max_seq_len, device):
    seq = list(prefix_ids)[-(max_seq_len - 4):]
    gen = []
    for _ in range(max_new_tokens):
        if len(seq) >= max_seq_len: break
        logits = model(torch.tensor([seq], dtype=torch.long, device=device))[:, -1, :]
        nid = int(torch.argmax(logits, dim=-1).item())
        seq.append(nid); gen.append(nid)
        if nid in stop_ids: break
    return gen

def _extract_answers_and_format_ok(tokens, mode):
    in_ans, in_t, preds = False, False, []
    saw_s, saw_e, t_cnt = False, False, 0
    for tok in tokens:
        if tok == '<ans>': in_ans = True; saw_s = True; continue
        if tok == '</ans>': saw_e = True; break
        if not in_ans: continue
        if tok == '<t>': in_t = True; t_cnt += 1; continue
        if tok == '</t>': in_t = False; continue
        if not in_t: preds.append(tok)
    if mode == 'turn_cot':
        fmt_ok = int(saw_s and saw_e and len(preds) == len(ANSWER_SLOTS))
    elif mode == 'stream_token_cot':
        fmt_ok = int(saw_s and saw_e and len(preds) == len(ANSWER_SLOTS) and t_cnt == len(ANSWER_SLOTS))
    else:
        fmt_ok = int(len(preds) == len(ANSWER_SLOTS))
    return preds[:len(ANSWER_SLOTS)], fmt_ok

@torch.no_grad()
def run_multiturn_inference(model, tokenizer, ep, mode, device, max_seq_len=1024):
    model.eval()
    stream_ids, turn_results, total_gen = [], [], 0
    end_turn_id, eos_id = tokenizer.end_turn_id, tokenizer.eos_token_id
    for t in ep.turns:
        stream_ids.extend(tokenizer.encode_tokens(t.prefix_tokens))
        prompt_len = len(stream_ids)
        if mode in ('turn_cot', 'stream_token_cot'):
            gen_ids = _greedy_generate_until(model, stream_ids, {end_turn_id, eos_id}, 80 if mode == 'turn_cot' else 90, max_seq_len, device)
            total_gen += len(gen_ids)
            stream_ids.extend(gen_ids)
            gen_toks = tokenizer.decode_tokens(gen_ids)
            preds, fmt_ok = _extract_answers_and_format_ok(gen_toks, mode)
            trace = ' '.join(gen_toks)
        else:
            stream_ids.append(tokenizer.ans_start_id)
            preds, traces, hop_fmt_ok = [], [], 1
            for _ in range(len(ANSWER_SLOTS)):
                gen_ids = _greedy_generate_until(model, stream_ids, {eos_id}, 20, max_seq_len, device)
                total_gen += len(gen_ids)
                gen_toks = tokenizer.decode_tokens(gen_ids)
                traces.append(' '.join(gen_toks))
                chosen = '<|unk|>'
                if '<t>' in gen_toks and '</t>' in gen_toks:
                    idx = gen_toks.index('</t>') + 1
                    if idx < len(gen_toks) and gen_toks[idx] != '<|eos|>': chosen = gen_toks[idx]
                    else: hop_fmt_ok = 0
                else:
                    hop_fmt_ok = 0
                    if gen_toks: chosen = gen_toks[0]
                preds.append(chosen)
                stream_ids.append(tokenizer.token_to_id.get(chosen, tokenizer.unk_token_id))
            stream_ids.extend([tokenizer.ans_end_id, end_turn_id])
            fmt_ok = hop_fmt_ok
            trace = ' || '.join(traces)
        gt = t.answers
        padded = preds + [''] * max(0, len(gt) - len(preds))
        hits = [int(padded[i] == gt[i]) for i in range(len(gt))]
        stale_hit = int(any(t.stale_answers[i] != gt[i] and padded[i] == t.stale_answers[i] for i in range(len(gt)))) if t.stale_answers else 0
        turn_results.append({'turn_idx': t.turn_idx, 'prompt_len': prompt_len, 'exact': int(all(hits)), 'format_ok': fmt_ok, 'hits': hits, 'stale_hit': stale_hit, 'preds': preds, 'gt': gt, 'trace': trace})
    return {'turns': turn_results, 'final_stream_len': len(stream_ids), 'total_gen': total_gen}

def evaluate_suite(model, tokenizer, mode, device, eval_episodes=25, max_seq_len=1024, seed=9999):
    specs = {
        'single_turn_d4': dict(num_turns=1, num_distractors=4, updates_per_turn=1),
        'long_range_d16': dict(num_turns=1, num_distractors=16, updates_per_turn=1),
        'ultra_long_d28': dict(num_turns=1, num_distractors=28, updates_per_turn=1),
        'multi_turn_3t_d6': dict(num_turns=3, num_distractors=6, updates_per_turn=1),
        'multi_turn_6t_d8': dict(num_turns=6, num_distractors=8, updates_per_turn=1),
    }
    results = {}
    for bname, bcfg in specs.items():
        rng = random.Random(seed + sum(ord(c) for c in bname))
        t0 = time.time()
        nt = bcfg['num_turns']
        t_exact, t_fmt, t_slot, t_y5, t_stale, t_plen = [0]*nt, [0]*nt, [0.0]*nt, [0]*nt, [0]*nt, [0.0]*nt
        flen_sum, gen_sum, shown = 0.0, 0.0, False
        for _ in range(eval_episodes):
            ep = generate_multiturn_episode(rng, **bcfg)
            out = run_multiturn_inference(model, tokenizer, ep, mode, device, max_seq_len)
            flen_sum += out['final_stream_len']; gen_sum += out['total_gen']
            for tr in out['turns']:
                ti = tr['turn_idx']
                t_exact[ti] += tr['exact']; t_fmt[ti] += tr['format_ok']
                t_slot[ti] += sum(tr['hits']) / len(ANSWER_SLOTS); t_y5[ti] += tr['hits'][-1]
                t_stale[ti] += tr['stale_hit']; t_plen[ti] += tr['prompt_len']
            if not shown and bname == 'multi_turn_6t_d8':
                shown = True; lt = out['turns'][-1]
                print(f"  [Sample {mode} | 6T Turn 6] prompt_len={lt['prompt_len']} | exact={bool(lt['exact'])} | fmt={bool(lt['format_ok'])}")
                print(f"    GT:    {lt['gt']}")
                print(f"    Pred:  {lt['preds']}")
                print(f"    Trace: {lt['trace'][:220]}")
        n = max(1, eval_episodes)
        per_turn = {f'turn_{i+1}': {'exact_acc': round(t_exact[i]/n, 3), 'format_ok': round(t_fmt[i]/n, 3), 'stale_rate': round(t_stale[i]/n, 3), 'avg_prompt_len': round(t_plen[i]/n, 1)} for i in range(nt)}
        results[bname] = {
            'overall_exact_acc': round(sum(t_exact)/(n*nt), 3),
            'overall_format_ok': round(sum(t_fmt)/(n*nt), 3),
            'overall_slot_acc': round(sum(t_slot)/(n*nt), 3),
            'overall_y5_acc': round(sum(t_y5)/(n*nt), 3),
            'overall_stale_rate': round(sum(t_stale)/(n*nt), 3),
            'avg_final_stream_len': round(flen_sum/n, 1),
            'avg_total_gen_tokens': round(gen_sum/n, 1),
            'eval_seconds': round(time.time()-t0, 1),
            'per_turn': per_turn,
        }
    return results

In [ ]:
# 5. 全3方式の学習 & 同一ベンチマーク評価を実行
if torch.cuda.is_available():
    device = torch.device('cuda')
    major, _ = torch.cuda.get_device_capability()
    amp_dtype = torch.bfloat16 if (major >= 8 and torch.cuda.is_bf16_supported()) else (torch.float16 if major >= 7 else torch.float32)
else:
    device, amp_dtype = torch.device('cpu'), torch.float32

model_cfg = {
    'num_tokens': tokenizer.vocab_size,
    'dim': cfg['dim'], 'depth': cfg['depth'], 'segment_len': cfg['segment_len'],
    'num_persist_mem_tokens': cfg['num_persist_mem_tokens'],
    'num_longterm_mem_tokens': cfg['num_longterm_mem_tokens'],
    'seq_len': cfg['seq_len'],
}
print(f'Device={device}, amp_dtype={amp_dtype}, model_cfg={model_cfg}')

all_metrics = {'preset': PRESET, 'model_type': MODEL_TYPE, 'model_cfg': model_cfg, 'stage1_ratio': STAGE1_RATIO, 'modes': {}}
for mode in MODES:
    m_steps = int(round(SFT_STEPS * EPHEMERAL_STEP_MULT)) if mode == 'ephemeral_token_cot' else SFT_STEPS
    model, tr_stats = train_mode(
        mode, MODEL_TYPE, model_cfg, tokenizer, device, amp_dtype,
        steps=m_steps, stage1_ratio=STAGE1_RATIO,
        batch_size=cfg['batch_size'], grad_accum=cfg['grad_accum'],
        lr=cfg['lr'], log_every=LOG_EVERY, seed=SEED,
    )
    ev_stats = evaluate_suite(model, tokenizer, mode, device, eval_episodes=EVAL_EPISODES, max_seq_len=cfg['seq_len'], seed=SEED + 9999)
    all_metrics['modes'][mode] = {'train': tr_stats, 'eval': ev_stats}
    print(f'\n[{mode}] Eval Result:')
    print(json.dumps(ev_stats, ensure_ascii=False, indent=2))
    del model
    if device.type == 'cuda': torch.cuda.empty_cache()

(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(all_metrics, ensure_ascii=False, indent=2), encoding='utf-8')

In [ ]:
# 6. 3方式の最終比較サマリー表示
print('=== Experiment 3.5: Format Warmup + Length Curriculum Summary ===')
header = f"{'Mode':20s} | {'d4':>5s} | {'d16':>5s} | {'d28':>5s} | {'3T':>5s} | {'6T':>5s} | {'6T_T6':>5s} | {'6T_fmt':>6s} | {'6T_len':>6s} | {'stale':>5s}"
print(header)
print('-' * len(header))
for mode in MODES:
    ev = all_metrics['modes'][mode]['eval']
    print(
        f"{mode:20s} | "
        f"{ev['single_turn_d4']['overall_exact_acc']:5.3f} | "
        f"{ev['long_range_d16']['overall_exact_acc']:5.3f} | "
        f"{ev['ultra_long_d28']['overall_exact_acc']:5.3f} | "
        f"{ev['multi_turn_3t_d6']['overall_exact_acc']:5.3f} | "
        f"{ev['multi_turn_6t_d8']['overall_exact_acc']:5.3f} | "
        f"{ev['multi_turn_6t_d8']['per_turn']['turn_6']['exact_acc']:5.3f} | "
        f"{ev['multi_turn_6t_d8']['overall_format_ok']:6.3f} | "
        f"{ev['multi_turn_6t_d8']['avg_final_stream_len']:6.1f} | "
        f"{ev['multi_turn_6t_d8']['overall_stale_rate']:5.3f}"
    )